<center><h1>Kotamraju_KrishnaKarthik_HW3</h1></center>


Name: Krishna Karthik Kotamraju  
Github Username: karthik05k9
USC ID: 9984105385


## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports (NumPy, pandas, and Python's standard library).


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


The AReM data are stored in `Homework 3 Data/AReM` within this folder. Each CSV file is one activity instance. There are seven activity folders and 88 files in total. The code below handles the space-separated `bending2/dataset4.csv` file while reading the data.


In [2]:
data_dir = Path('Homework 3 Data/AReM')
activities = ['bending1', 'bending2', 'cycling', 'lying',
              'sitting', 'standing', 'walking']
sensor_columns = ['avg_rss12', 'var_rss12', 'avg_rss13',
                  'var_rss13', 'avg_rss23', 'var_rss23']
print('Activity folders:', len(activities))
print('CSV files:', sum(len(list((data_dir / a).glob('dataset*.csv')))
                        for a in activities))


Activity folders: 7
CSV files: 88


### (b) Test and Train Data

In [3]:
instances = []
for activity in activities:
    files = sorted((data_dir / activity).glob('dataset*.csv'),
                   key=lambda path: int(path.stem.replace('dataset', '')))
    for file in files:
        number = int(file.stem.replace('dataset', ''))
        test_numbers = (1, 2) if activity.startswith('bending') else (1, 2, 3)
        split = 'test' if number in test_numbers else 'train'
        separator = r'\s+' if activity == 'bending2' and number == 4 else ','
        data = pd.read_csv(file, skiprows=5, header=None, sep=separator,
                           names=['time'] + sensor_columns, usecols=range(7))
        instances.append({'activity': activity, 'file': file.name,
                          'split': split, 'data': data})

split_counts = pd.crosstab(
    pd.Series([item['activity'] for item in instances], name='Activity'),
    pd.Series([item['split'] for item in instances], name='Split'))
display(split_counts)
print('Training instances:', sum(item['split'] == 'train' for item in instances))
print('Test instances:', sum(item['split'] == 'test' for item in instances))
print('Files with fewer than 480 readings:',
      [(item['activity'], item['file'], len(item['data']))
       for item in instances if len(item['data']) != 480])


Split,test,train
Activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Training instances: 69
Test instances: 19
Files with fewer than 480 readings: [('sitting', 'dataset8.csv', 479)]


### (c) Feature Extraction

#### i. Research

Common time-domain features include minimum and maximum (extreme values), mean and median (typical values), standard deviation and range (spread), first and third quartiles and interquartile range (distribution), and autocorrelation (how values relate to earlier values). Skewness can also describe whether a series is asymmetric. These are examples of features that summarize the values or their order over time; only the seven features requested in part ii are calculated below.


#### ii. Extraction

In [4]:
feature_rows = []
instance_info = []
for item in instances:
    row = {}
    for number, column in enumerate(sensor_columns, start=1):
        values = item['data'][column]
        row[f'min{number}'] = values.min()
        row[f'max{number}'] = values.max()
        row[f'mean{number}'] = values.mean()
        row[f'median{number}'] = values.median()
        row[f'std{number}'] = values.std()
        row[f'q1_{number}'] = values.quantile(0.25)
        row[f'q3_{number}'] = values.quantile(0.75)
    feature_rows.append(row)
    instance_info.append((item['activity'], item['file'], item['split']))

index = pd.Index(range(1, len(feature_rows) + 1), name='Instance')
features = pd.DataFrame(feature_rows, index=index)
instance_info = pd.DataFrame(instance_info, index=index,
                             columns=['Activity', 'File', 'Split'])
print('Feature table shape:', features.shape)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(features.round(3))


Feature table shape: (88, 42)


,min1,max1,mean1,median1,std1,q1_1,q3_1,min2,max2,mean2,median2,std2,q1_2,q3_2,min3,max3,mean3,median3,std3,q1_3,q3_3,min4,max4,mean4,median4,std4,q1_4,q3_4,min5,max5,mean5,median5,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.625,40.500,1.477,39.250,42.000,0.0,1.30,0.359,0.430,0.323,0.000,0.500,4.00,29.50,19.041,19.250,4.463,16.000,23.250,0.0,7.23,0.833,0.500,0.966,0.000,1.120,27.25,38.25,34.311,35.000,2.188,33.000,36.000,0.00,1.92,0.571,0.430,0.583,0.000,1.300
2,38.00,45.67,42.813,42.500,1.436,42.000,43.670,0.0,1.22,0.372,0.470,0.289,0.000,0.500,2.00,29.50,20.097,21.000,3.894,19.000,22.250,0.0,5.76,0.876,0.500,1.047,0.000,1.145,27.67,38.50,33.025,33.000,1.995,32.000,34.500,0.00,3.11,0.571,0.430,0.601,0.000,1.300
3,35.00,47.40,43.954,44.330,1.559,43.000,45.000,0.0,1.70,0.426,0.470,0.339,0.000,0.500,6.50,29.75,22.122,23.000,3.031,19.750,24.000,0.0,4.44,0.497,0.430,0.551,0.000,0.830,29.00,38.50,35.588,36.000,2.000,35.362,36.500,0.00,1.79,0.493,0.430,0.514,0.000,0.940
4,33.00,47.75,42.180,43.500,3.671,39.150,45.000,0.0,3.00,0.696,0.500,0.631,0.000,1.120,8.50,30.00,22.184,23.000,3.810,20.500,24.372,0.0,5.15,0.990,0.830,0.954,0.430,1.300,20.00,38.67,33.494,35.000,3.849,30.458,36.330,0.00,2.18,0.614,0.500,0.524,0.000,1.000
5,33.00,45.75,41.678,41.750,2.243,41.330,42.750,0.0,2.83,0.536,0.500,0.405,0.430,0.710,3.00,28.25,19.007,19.125,4.087,16.500,22.062,0.0,6.42,0.842,0.500,0.929,0.430,1.120,23.67,37.50,29.857,30.000,2.411,28.458,31.250,0.00,1.79,0.383,0.430,0.389,0.000,0.500
6,37.00,48.00,43.455,43.250,1.386,42.500,45.000,0.0,1.58,0.378,0.470,0.316,0.000,0.500,5.75,27.00,15.793,15.000,3.848,13.000,18.270,0.0,10.03,0.849,0.500,0.996,0.430,1.120,8.00,33.50,23.035,23.500,2.489,22.250,24.000,0.00,5.26,0.680,0.500,0.623,0.430,0.870
7,36.25,48.00,43.969,44.500,1.618,43.310,44.670,0.0,1.50,0.413,0.470,0.263,0.430,0.500,1.50,26.33,15.868,16.250,3.742,14.250,18.000,0.0,5.17,0.666,0.470,0.789,0.000,0.940,11.33,30.75,22.104,21.670,3.318,20.500,23.750,0.00,2.96,0.555,0.490,0.488,0.000,0.830
8,12.75,51.00,24.563,24.250,3.738,23.188,26.500,0.0,6.87,0.591,0.430,0.837,0.000,0.710,0.00,25.33,19.121,20.250,4.300,18.000,22.000,0.0,6.76,0.737,0.470,0.927,0.000,0.870,10.67,30.00,23.493,23.750,3.694,20.500,27.000,0.00,4.97,0.700,0.500,0.694,0.430,0.870
9,0.00,42.75,27.465,28.000,3.584,25.500,30.000,0.0,7.76,0.450,0.430,0.767,0.000,0.500,7.50,35.00,20.843,20.750,3.826,18.000,22.500,0.0,5.76,0.780,0.500,0.904,0.000,0.940,0.00,33.00,17.618,18.000,5.054,15.000,20.750,0.00,6.76,1.122,0.830,1.012,0.470,1.300


#### iii. Standard Deviation and 90% Bootstrap Confidence Intervals

For each of the 42 columns, the standard deviation is calculated across the 88 instances. A bootstrap sample draws 88 instances with replacement; the 5th and 95th percentiles of 1,000 bootstrap estimates give a 90% percentile confidence interval. The sample standard deviation (`ddof=1`) is used for both the estimate and the bootstrap.


In [5]:
rng = np.random.default_rng(552)
results = []
for feature in features.columns:
    values = features[feature].to_numpy()
    bootstrap_stds = []
    for _ in range(1000):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_stds.append(np.std(sample, ddof=1))
    lower, upper = np.percentile(bootstrap_stds, [5, 95])
    results.append((feature, features[feature].std(), lower, upper))

std_intervals = pd.DataFrame(
    results, columns=['Feature', 'Std. Dev.', '90% Lower', '90% Upper']
).set_index('Feature')
display(std_intervals.round(3))


,Std. Dev.,90% Lower,90% Upper
Feature,,,
min1,9.570,8.235,10.689
max1,4.394,3.390,5.220
mean1,5.336,4.700,5.874
median1,5.440,4.817,6.005
std1,1.772,1.582,1.945
q1_1,6.154,5.594,6.658
q3_1,5.139,4.299,5.814
min2,0.000,0.000,0.000
max2,5.063,4.629,5.398


#### iv. Select Features

I would choose **mean, standard deviation, and maximum**. The mean describes a typical reading, standard deviation describes how much the readings change, and maximum captures a high value that the mean may hide. These three describe different aspects of an activity.


## 2. ISLR 3.7.4

### (a) Linear Train

**Cubic training RSS is less than or equal to linear training RSS.** A cubic model includes the linear model as a special case (its squared and cubed coefficients can be zero), so fitting it by least squares cannot produce a larger training RSS.


### (b) Linear Test

**The linear model is expected to have lower test RSS.** When the true relationship is linear, the extra terms in the cubic model can fit noise in the training data.


### (c) Not Linear Train

**Cubic training RSS is still less than or equal to linear training RSS.** The cubic model is more flexible on the training observations, regardless of the shape of the true relationship.


### (d) Not Linear Testing

**There is not enough information to choose the lower test RSS.** If the true relationship is only slightly nonlinear, the simpler model could predict better; if it is well approximated by a cubic, the cubic model could predict better. The result also depends on the amount of data and noise.

### References
- [AReM data, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/366/activity+recognition+system+based+on+multisensor+data+fusion+arem)
- James et al., *An Introduction to Statistical Learning*, Exercise 3.7.4.
